In [ ]:
# ==========================================================================
# Diagnostic Percept — 02 · Benchmark (H6 + consensus-flip)
# Run the MedQA-USMLE test set (1273 questions) under all intervention
# conditions, multi-GPU data-parallel on 4× A100, then the consensus-flip
# enrichment analysis. The expensive phase (~2 h on 4× A100). Fully
# resumable: re-run after a disconnect and it picks up from the jsonls.
# Run phases in order 00 -> 01 -> 02 -> 03 -> 04, each a separate Colab
# Enterprise runtime. State is shared through results/ mirrored to a GCS
# bucket (set GCS_BUCKET) or Drive. Qwen3 only (32B/14B/8B/4B by GPU mem).
# ==========================================================================
print('=== Diagnostic Percept | 02 Benchmark (H6 + consensus-flip) ===')

In [ ]:
# Boot disk on Vertex AI Colab Enterprise is ~101 GB and starts ~60 GB
# full (system image). /content is the 527 GB workspace. Without
# redirection, pip's temp build files + pip cache + HF cache all land
# on the boot disk and can fill it during the install — at which point
# Vertex AI health checks fail and the runtime is marked unhealthy.
# Set EVERY cache dir to /content BEFORE the first pip call.
import os, subprocess, sys, shutil
from pathlib import Path
_C = Path('/content/.cache') if Path('/content').exists() else None
if _C:
    _C.mkdir(parents=True, exist_ok=True)
    (_C / 'pip').mkdir(exist_ok=True)
    (_C / 'tmp').mkdir(exist_ok=True)
    os.environ['PIP_CACHE_DIR']  = str(_C / 'pip')
    os.environ['TMPDIR']         = str(_C / 'tmp')
    os.environ['HF_HOME']        = str(_C / 'huggingface')
    os.environ['HF_HUB_CACHE']   = str(_C / 'huggingface')
    os.environ['TRANSFORMERS_CACHE'] = str(_C / 'transformers')
    os.environ['TORCH_HOME']     = str(_C / 'torch')
    os.environ['XDG_CACHE_HOME'] = str(_C)
    print(f'Caches → {_C} (boot disk is small; this is mandatory)')

def _disk(label=''):
    for p in ('/', '/content'):
        if Path(p).exists():
            s = shutil.disk_usage(p)
            free = (s.total - s.used) / 1e9
            print(f'  [{label}] disk {p:<10} free={free:6.1f} GB')
_disk('start')

# Surgical upgrade: install transformers main with --no-deps so it does
# NOT pull a newer torch / torchvision / pillow. Then pin transformers'
# runtime deps to the *exact* versions it expects (it pins tokenizers
# <=0.23.0, which a bare `--upgrade tokenizers` overshoots to 0.23.1).
def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

# 1. transformers main, no cascading dep upgrades
_pip('--upgrade', '--no-deps',
     'transformers @ git+https://github.com/huggingface/transformers.git@main')
# 2. transformers' runtime deps. huggingface_hub MUST come from main
#    too — transformers main imports `is_offline_mode` which only
#    exists in hub's main branch (older released versions removed it,
#    newer renamed it). Install hub from git@main to match.
_pip('--no-deps', '--upgrade',
     'huggingface_hub @ git+https://github.com/huggingface/huggingface_hub.git@main',
     'safetensors>=0.4',
     'tokenizers>=0.22.0,<=0.23.0',
     'regex',
     'requests',
     'pyyaml',
     'httpx',
     'filelock')
# 3. our other libs --no-deps (accelerate / bitsandbytes happy w/ Colab torch)
_pip('--upgrade', '--no-deps', 'accelerate>=0.34', 'bitsandbytes>=0.43')
# 4. plain installs of small libs (no risk to torch/pillow)
_pip('scikit-learn', 'matplotlib', 'tqdm', 'datasets', 'nbformat', 'ipywidgets')
_disk('after step 4')
# 5. Pillow self-heal if a prior run pulled pillow 12 (PIL.ImageText breaks).
try:
    import PIL.ImageText  # canary for pillow 12 ABI break
except Exception:
    print('Repairing pillow (pinning <12) ...')
    _pip('--force-reinstall', '--no-deps', 'pillow<12')

# Free pip cache to reclaim disk now that everything is installed.
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False, capture_output=True)
_disk('after purge')

# Drop the pre-imported transformers + huggingface_hub from Colab so the
# re-import picks up the new versions.
import importlib
for m in [k for k in list(sys.modules)
          if k in ('transformers', 'huggingface_hub')
          or k.startswith('transformers.') or k.startswith('huggingface_hub.')]:
    del sys.modules[m]
importlib.invalidate_caches()

# Self-heal: if the import still fails because hub<->transformers got
# out of sync, re-install both from main and retry once.
try:
    import transformers
except ImportError as _e:
    print(f'Self-healing transformers/hub mismatch: {_e}')
    _pip('--no-deps', '--upgrade', '--force-reinstall',
         'transformers @ git+https://github.com/huggingface/transformers.git@main',
         'huggingface_hub @ git+https://github.com/huggingface/huggingface_hub.git@main')
    for m in [k for k in list(sys.modules)
              if k in ('transformers', 'huggingface_hub')
              or k.startswith('transformers.') or k.startswith('huggingface_hub.')]:
        del sys.modules[m]
    importlib.invalidate_caches()
    import transformers
_has_q35 = hasattr(transformers, 'Qwen3_5ForCausalLM')
print(f'transformers {transformers.__version__}  Qwen3_5 registered: {_has_q35}')
if not _has_q35:
    # NB: do NOT auto-restart the kernel here. Vertex AI's idle detector
    # interprets the post-restart wait as inactivity and may shut the VM
    # down within minutes. Instead, halt cleanly with a clear message so
    # the user does the restart manually and immediately Run All again.
    raise SystemExit(
        '\n' + '=' * 70 +
        '\n  ACTION REQUIRED: restart the kernel, then click Run All again.'
        '\n  Colab Enterprise: Runtime → Restart session → Run all.'
        '\n  (Auto-restart removed because Vertex AI counts the post-'
        '\n   restart idle time toward the auto-shutdown timer.)'
        '\n' + '=' * 70
    )

In [ ]:
# === EMERGENCY DISK CLEANUP — uncomment, run, re-comment ====================
# Use when `df -h /` shows < 10 GB free on the boot disk after the install
# step. Each block is independent; you can run just one or all of them.
#
# import subprocess, shutil, gc, os
# from pathlib import Path
#
# # 1. Boot-disk caches (the usual offenders).
# for d in ('/root/.cache/pip', '/root/.cache/huggingface',
#           '/root/.cache/torch', '/root/.cache/matplotlib',
#           '/root/.cache/black', '/root/.triton'):
#     subprocess.run(['rm', '-rf', d], check=False)
# # 2. /tmp leftovers (pip build dirs, torch inductor, model shards).
# for pattern in ('/tmp/pip*', '/tmp/torch*', '/tmp/cuda*', '/tmp/hf*'):
#     subprocess.run(f'rm -rf {pattern}', shell=True, check=False)
# # 3. The pip download cache (~/ + system).
# subprocess.run(['python', '-m', 'pip', 'cache', 'purge'], check=False)
# # 4. Stale HuggingFace lockfiles on the workspace (rare; safe to clear).
# subprocess.run(['find', '/content/.cache/huggingface', '-name', '*.lock',
#                 '-delete'], check=False)
# # 5. Old results from a prior run on /content (only if you don't need them).
# # shutil.rmtree('/content/results', ignore_errors=True)
# # 6. CUDA allocator + Python garbage. Releases any held GPU mem.
# gc.collect()
# try:
#     import torch
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         for i in range(torch.cuda.device_count()):
#             torch.cuda.reset_peak_memory_stats(i)
# except Exception:
#     pass
# for p in ('/', '/content'):
#     if Path(p).exists():
#         s = shutil.disk_usage(p)
#         print(f'  disk {p:<10} free={(s.total-s.used)/1e9:6.1f} GB')
# ============================================================================

In [ ]:
import os
from pathlib import Path
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# Redirect HF / Torch caches to /content (Colab Enterprise's 195 GB workspace
# disk) so model weights don't fill the ~90 GB boot disk. Must happen before
# transformers / huggingface_hub are imported, so set it here.
_CACHE_ROOT = '/content/.cache' if Path('/content').exists() else None
if _CACHE_ROOT:
    Path(_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME',          f'{_CACHE_ROOT}/huggingface')
    os.environ.setdefault('TRANSFORMERS_CACHE', f'{_CACHE_ROOT}/transformers')
    os.environ.setdefault('TORCH_HOME',       f'{_CACHE_ROOT}/torch')
    os.environ.setdefault('XDG_CACHE_HOME',   _CACHE_ROOT)
    print(f'Caches redirected to {_CACHE_ROOT}')
else:
    print('No /content workspace (not on Colab); using default cache dirs.')

# Force tqdm.notebook so progress bars render as Colab widgets, not raw lines
# (matters for the long H6/H7/sycophancy passes).
try:
    import tqdm, tqdm.notebook
    tqdm.tqdm = tqdm.notebook.tqdm
    import tqdm.auto
    tqdm.auto.tqdm = tqdm.notebook.tqdm
    print('tqdm.notebook installed as the default tqdm')
except Exception as _e:
    print('tqdm.notebook unavailable, keeping default:', _e)

In [ ]:
# === env check ===
import traceback
try:

    import os, sys, subprocess, json, time, traceback, importlib
    from pathlib import Path
    import torch

    # Runtime detection — free Colab vs Colab Enterprise (Vertex Workbench) vs other.
    def _detect_runtime():
        if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
            try:
                import google.colab  # noqa: F401
                return 'colab_free'
            except ImportError:
                pass
        if any(k in os.environ for k in ('GOOGLE_CLOUD_PROJECT', 'VERTEX_PRODUCT')):
            return 'colab_enterprise'
        if 'JUPYTERHUB_USER' in os.environ:
            return 'jupyterhub'
        return 'local'
    RUNTIME = _detect_runtime()
    print(f'Runtime: {RUNTIME}')
    print('Python:', sys.version.split()[0])
    print('Torch :', torch.__version__)
    print('CUDA  :', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        gpu_name = torch.cuda.get_device_name(0)
        print(f'GPU: {gpu_name}  | Memory: {gpu_gb:.1f} GB')

        # NOTE: do NOT set MODEL_OVERRIDE / USE_4BIT / N_BENCH here. All
        # decisioning lives in src/setup.py auto_pick(), which is called by
        # smart_load_model() in the model-load cell. If you set env vars
        # here, an old snapshot of THIS cell (frozen in your imported .ipynb)
        # could write a stale Qwen3.5/3.6 pick that auto_pick can't override
        # because the env var "wins". src/setup.py also actively strips
        # MODEL_OVERRIDE if it points to a known-broken Qwen3.5/3.6 checkpoint.

    # Disk sanity. Colab Enterprise's boot disk is ~94 GB and starts ~90% full
    # (system image). /content is the 195 GB workspace where caches go.
    import shutil
    for path in ('/', '/content'):
        if Path(path).exists():
            s = shutil.disk_usage(path)
            used_pct = 100 * s.used / s.total
            warn = ' !! LOW' if (s.total - s.used) < 10 * (1024**3) else ''
            print(f'Disk {path:<10}  {s.used/1e9:6.1f} / {s.total/1e9:6.1f} GB  ({used_pct:.0f}%){warn}')

    # Validate cache redirect — the model download (~14 GB at NF4, ~54 GB at bf16)
    # MUST land on /content or the boot disk fills up.
    _hf_home = os.environ.get('HF_HOME', '')
    if _hf_home and not _hf_home.startswith('/content'):
        print('!! WARN: HF_HOME is', _hf_home, '— model will download to boot disk!')
    elif _hf_home:
        print(f'HF cache → {_hf_home}  (/content has plenty of room)')
    else:
        print('!! WARN: HF_HOME not set; model download will use ~/.cache (boot disk).')

    REPO_URL = 'https://github.com/ArioMoniri/diagnosticpercept.git'
    REPO_DIR = 'diagnosticpercept'
    if not Path(REPO_DIR).exists():
        print('Cloning', REPO_URL, '...')
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        # Hard-reset to origin/main so re-runs always pick up the latest code.
        print('Fetching + hard-resetting to origin/main ...')
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=False)

    # Print current SHA so we can verify the running version.
    sha = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print(f'Repo @ commit: {sha}  (expect 8635794 or newer for H4+H5)')

    # Drop any previously-imported src.* modules so Python re-loads from disk —
    # a kernel re-run with the prior clone may have cached the old discover.py.
    for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
        del sys.modules[m]
    importlib.invalidate_caches()

    repo_path = str(Path(REPO_DIR).resolve())
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    RESULTS = Path('/content/results'); RESULTS.mkdir(parents=True, exist_ok=True)
    print('Repo   :', repo_path)
    print('Results:', RESULTS)
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === preflight — print the run plan ===
import traceback
try:

    # Single-glance summary of what's about to happen so you can abort before
    # downloading 14 GB of model weights if anything is wrong.
    print('=' * 62)
    print(f'  Runtime       : {RUNTIME}')
    _n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if _n_gpu == 0:
        print('  GPU           : none (CPU only)')
    else:
        # PER-GPU memory report — important on 4× A100 because nvidia-smi may
        # show a heterogeneous mix if one GPU was previously used by another
        # process (Vertex AI doesn't always reset cleanly across notebook runs).
        for i in range(_n_gpu):
            p = torch.cuda.get_device_properties(i)
            gb = p.total_memory / 1e9
            used = torch.cuda.memory_allocated(i) / 1e9
            print(f'  GPU{i}          : {p.name}  total={gb:.1f} GB  '
                  f'currently_allocated={used:.2f} GB')
    print(f'  Model         : {os.environ.get("MODEL_OVERRIDE", "(auto-pick from chain)")}')
    print(f'  Quantize 4bit : {os.environ.get("USE_4BIT", "auto")}')
    print(f'  N_BENCH       : {os.environ.get("N_BENCH", "default")}')
    print(f'  HF cache      : {os.environ.get("HF_HOME", "(default ~/.cache)")}')
    print(f'  CUDA alloc    : {os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "(unset)")}')
    print()
    print('  Estimated wall time on this hardware:')
    _gpu_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if _n_gpu else 0
    _gpu_name = torch.cuda.get_device_name(0) if _n_gpu else ''
    # H100 ≈ 1.5× A100 fwd throughput. Wall time scales by 1/n_gpu for H6.
    _is_h100 = 'H100' in _gpu_name
    _throughput_factor = 1.0 if _is_h100 else 1.5  # A100 vs H100
    _par = max(1, _n_gpu)
    print(f'  Hardware: {_n_gpu}× {_gpu_name or "CPU"}  (parallel factor {_par})')
    if _gpu_gb >= 36:
        h1   = round(5  * _throughput_factor, 1)             # H1 stays single-GPU
        h6   = round(75 * _throughput_factor / _par, 1)      # parallelized
        h7   = round(6  * _throughput_factor, 1)             # single-GPU
        syc  = round(15 * _throughput_factor, 1)             # single-GPU for now
        total = h1 + h6 + h7 + syc
        print(f'    H1 discover           ~{h1} min  (single GPU)')
        print(f'    H6 deep (1273×6)      ~{h6} min  ({_par}× parallel)')
        print(f'    H7 (300 items)        ~{h7} min  (single GPU)')
        print(f'    H8 + sycophancy       ~{syc} min  (single GPU)')
        print(f'    --- TOTAL             ~{total/60:.1f} hr')
    else:
        print('    Small-GPU budget — auto-pick will drop model size.')
    print('=' * 62)

    # On 4-GPU machines the data-parallel H6 worker holds an extra ~3 GB cuBLAS
    # workspace per device by default. Setting CUBLAS_WORKSPACE_CONFIG=:0:0
    # disables that pool (we don't need deterministic cuBLAS for inference) and
    # saves ~12 GB across 4 GPUs — buys back the H6 KV cache headroom on A100-40.
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':0:0')

    # expandable_segments cuts fragmentation across the many small allocs the
    # H6 reasoning chain produces (every gen.scores entry is its own alloc).
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === HF login (optional — only for gated models) ===
import traceback
try:

    import os
    # Qwen3 is open-weights and needs NO token. HF_TOKEN is only needed if you
    # override to a gated model. Resolution order:
    #   1. env var HF_TOKEN
    #   2. Colab secret HF_TOKEN
    #   3. interactive notebook_login() widget (may not render in all Colab
    #      runtimes — if so, use the manual paste cell that follows)
    def _resolve_hf_token():
        if os.environ.get('HF_TOKEN'):
            print('HF_TOKEN already set in env.')
            return
        # Free Colab has google.colab.userdata; Colab Enterprise does NOT.
        if RUNTIME == 'colab_free':
            try:
                from google.colab import userdata
                tok = userdata.get('HF_TOKEN')
                if tok:
                    os.environ['HF_TOKEN'] = tok
                    print('HF_TOKEN loaded from Colab secret.')
                    return
            except Exception:
                pass
        print('No HF_TOKEN in env.')
        if RUNTIME == 'colab_enterprise':
            print('Colab Enterprise: set HF_TOKEN as a runtime-template env var,')
            print('or paste into the manual cell below.')
        else:
            print('Qwen3 is open-weights so this is fine to skip for the default chain.')
        try:
            from huggingface_hub import notebook_login
            notebook_login()
            print('Token widget rendered above ↑ (paste + Login).')
            print('If you do not see a widget, use the manual paste cell below.')
        except Exception as e:
            print(f'(notebook_login unavailable: {e})')

    _resolve_hf_token()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === HF token: manual paste fallback (skip if widget worked) ===
import traceback
try:

    # If the widget above didn't render, paste your token below between the quotes
    # and run THIS cell. Leave blank to skip.
    HF_TOKEN_PASTE = ''   # ← paste like 'hf_xxxxxxxxxxxxxxxxx', then Run cell

    if HF_TOKEN_PASTE.strip():
        os.environ['HF_TOKEN'] = HF_TOKEN_PASTE.strip()
        print(f'HF_TOKEN set manually ({len(HF_TOKEN_PASTE.strip())} chars).')
    else:
        print('No manual token pasted. Continuing with whatever the previous cell resolved.')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === load model ===
import traceback
try:

    # All decision logic lives in src/setup.py — fixes to GPU detection, model
    # auto-pick, or max_memory take effect on the next Run All without
    # re-importing the notebook (the env-check cell pulls latest src/ first).
    from src.setup import smart_load_model
    from src.model import set_seed
    set_seed(0)

    lm, MODEL_NAME = smart_load_model()
    # Legacy globals so downstream cells keep working.
    USE_4BIT = bool(int(os.environ.get('USE_4BIT', '0')))
    N_BENCH  = int(os.environ.get('N_BENCH', '1273'))
    n_gpus   = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === memory helpers — reclaim VRAM + disk between sections ===
import traceback
try:

    # Lightweight helpers we'll call between H1/H2/.../H8 to keep VRAM bounded.
    # H4-H7 each cache large activation tensors in Python globals; without an
    # explicit drop between sections the cuBLAS allocator's reserved pool
    # ratchets up and the H6 reasoning chain can OOM 90 min in.
    import gc, shutil
    from pathlib import Path

    def _free_vram(label=''):
        gc.collect()
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats(i)
            used = [torch.cuda.memory_allocated(i)/1e9
                    for i in range(torch.cuda.device_count())]
            print(f'  [free_vram {label}] alloc/GPU = ' +
                  ' '.join(f'{u:.2f}' for u in used) + ' GB')
        # /content disk free.
        if Path('/content').exists():
            s = shutil.disk_usage('/content')
            print(f'  [free_vram {label}] /content free = {(s.total-s.used)/1e9:.1f} GB')

    _free_vram('post-load')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === sanity: h.retain_grad flows ===
import traceback
try:

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    ids = lm.tokenizer('Chest pain. Diagnosis:', return_tensors='pt').input_ids.to(lm.device)
    lm.model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        out = lm.model(input_ids=ids, use_cache=False)
        # logit at last position only — no need for full vocab sum.
        out.logits[0, -1, 0].backward()
    g = lm.layers[0].mlp._h.grad
    assert g is not None, 'h.grad is None — hook patching failed.'
    assert torch.isfinite(g).all(), 'h.grad has non-finite values.'
    assert g.abs().sum() > 0, 'h.grad is all zeros.'
    print('OK: layer-0 h.grad shape', tuple(g.shape), 'nonzero =', (g.abs() > 0).sum().item())
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'  VRAM after sanity: {torch.cuda.memory_allocated()/1e9:.2f} GB')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === persist: restore results/ from the shared backend =====================
# Each phase runs in a SEPARATE Colab Enterprise runtime, so /content starts
# empty. To see the previous phase's artifacts (discovery.json, the H6 jsonls,
# comparison.csv …) we restore results/ from a shared backend chosen here.
#
# RECOMMENDED on Colab Enterprise: a GCS bucket. Set it once per runtime:
#     %env GCS_BUCKET=gs://your-bucket-name
# (gsutil is pre-installed and the runtime service account has access.)
# Free-Colab fallback: Google Drive is auto-mounted if no bucket is set.
import os, subprocess
from src.persist import detect_backend, build_sync_cmd, remote_location

_drive_ok = Path('/content/drive').exists()
if not os.environ.get('GCS_BUCKET') and not _drive_ok:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        _drive_ok = Path('/content/drive').exists()
    except Exception as _e:
        print('Drive mount unavailable (fine if you are using GCS):', _e)

BACKEND = detect_backend(os.environ, _drive_ok)
BUCKET  = os.environ.get('GCS_BUCKET') or None
REMOTE  = remote_location(BACKEND, bucket=BUCKET) if BACKEND != 'local' else None
print(f'Persistence backend = {BACKEND}   remote = {REMOTE}')

import shutil as _shutil
def _sync(src, dst, backend, label):
    """Run one rsync/gsutil sync, guarding a missing CLI + first-phase noise."""
    cmd = build_sync_cmd(src, dst, backend)
    if _shutil.which(cmd[0]) is None:
        print(f'!! {cmd[0]!r} not on PATH — cannot {label}. '
              f'On Colab Enterprise gsutil is preinstalled; for Drive, rsync is.')
        return
    print(f'{label}:', ' '.join(cmd))
    # capture_output so an empty-remote gsutil CommandException on phase-0
    # restore doesn't dump a scary multi-line stderr; surface it only if it
    # looks like a real failure.
    r = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if r.returncode != 0 and 'does not name a directory' not in (r.stderr or ''):
        tail = (r.stderr or '').strip().splitlines()[-3:]
        if tail:
            print('   (note)', ' | '.join(tail))

RESULTS.mkdir(parents=True, exist_ok=True)
if REMOTE:
    if BACKEND == 'drive':
        Path(REMOTE).mkdir(parents=True, exist_ok=True)
    # remote -> local. check=False semantics: on the FIRST phase the remote is
    # empty, which is not an error.
    _sync(REMOTE, str(RESULTS), BACKEND, 'restore')
    print('Restored results/ from', REMOTE)
else:
    print('!! local backend: this phase will NOT see other phases\' outputs.')
    print('!! Set GCS_BUCKET (recommended) or mount Drive to chain phases.')

In [ ]:
# === H6 — setup conditions from discovery checkpoint =======================
from src.healthbench import (load_medqa, run_conditions, ablate_neurons_factory,
                             anchor_factory, zero_mlp_factory)
from src.checkpoint import load_discovery

disc = load_discovery(RESULTS)   # raises a clear error if 01_discovery hasn't run
if disc.model_name != MODEL_NAME:
    print(f'!! WARNING: discovery used {disc.model_name} but this runtime loaded '
          f'{MODEL_NAME}.\n!! Neuron indices are model-specific — re-run '
          f'01_discovery.ipynb on THIS model if the two differ.')
L_star, N_star = disc.gate_layer, disc.gate_neuron
m_star, anchor_d = disc.gate_m_star, disc.gate_anchor_d
critical = disc.critical_layer
top_overconf = disc.overconf_neurons[:3]
top_halluc   = disc.halluc_neurons[:3]
combined     = disc.combined_neurons
N_BENCH = int(os.environ.get('N_BENCH', '1273'))
DATASET = 'GBaker/MedQA-USMLE-4-options-hf'
items = load_medqa(DATASET, split='test', n=N_BENCH, seed=0)
print(f'Restored discovery (gate L{L_star}:F{N_star} m*={m_star} d={anchor_d:.4f}, '
      f'critical L{critical}); loaded {len(items)} MedQA items.')

H6_RESULTS = RESULTS / 'h6'; H6_RESULTS.mkdir(exist_ok=True)
_N_GPUS_PRE = torch.cuda.device_count() if torch.cuda.is_available() else 0
H6_MODE = 'DEEP_FULL' if _N_GPUS_PRE >= 4 else 'DEEP'   # 4× GPUs → all 6 conditions

ALL_CONDITIONS = {
    'baseline':           None,
    'h1_gate_anchor':     anchor_factory(lm.layers, L_star, N_star, m_star, anchor_d, k=1.0),
    'h3_zero_layer':      zero_mlp_factory(lm.layers, [critical]),
    'h4_ablate_halluc':   ablate_neurons_factory(lm.layers, top_halluc),
    'h5_ablate_overconf': ablate_neurons_factory(lm.layers, top_overconf),
    'h4_h5_combined':     ablate_neurons_factory(lm.layers, combined),
}
DEEP_KEYS = ['baseline', 'h1_gate_anchor', 'h5_ablate_overconf']
CONDITIONS = ALL_CONDITIONS if H6_MODE in ('FAST', 'DEEP_FULL') else {k: ALL_CONDITIONS[k] for k in DEEP_KEYS}
print(f'Mode = {H6_MODE} → {len(CONDITIONS)} conditions × {len(items)} questions')

In [ ]:
# === H6 — wall-time probe (times 2 questions before the full run) ==========
# This prints an HONEST estimate so a slow config never silently eats 24 h.
# It runs on the main model (still loaded here, before the free-for-parallel
# cell). With the early-stop fix a question is ~a few seconds, not ~25 s.
import time as _time
from src.healthbench import run_one
_pn = min(2, len(items))
_t = _time.time()
for _it in items[:_pn]:
    _ = run_one(lm, _it, 'baseline')
_per_q = (_time.time() - _t) / max(1, _pn)
_ng = max(1, torch.cuda.device_count() if torch.cuda.is_available() else 1)
_est_h = _per_q * len(items) * len(CONDITIONS) / _ng / 3600
print(f'~{_per_q:.1f}s/question  →  est H6 wall-time {_est_h:.1f} h '
      f'for {len(items)} questions × {len(CONDITIONS)} conditions on {_ng} GPU(s)')
if _est_h > 3:
    print('!! >3 h. For a first pass set a smaller N and re-run THIS notebook:')
    print('!!     %env N_BENCH=300        (then Runtime → Run all)')
    print('!! The run is resumable, so you can scale N_BENCH back up later.')
del _t, _per_q, _est_h

In [ ]:
# === free the main model before the multi-GPU H6 run =======================
# On 4× A100-40 the data-parallel path spawns one worker per GPU, and EACH
# worker loads its own NF4 copy (~18 GB for Qwen3-32B). The main-process copy
# still sits on GPU0, so worker-0 would try to fit a second ~18 GB model on
# the same 40 GB card → OOM on long reasoning chains. Since the parallel path
# rebuilds every intervention from JSON specs inside the workers, the main
# copy is dead weight here — drop it (and the lm-bound CONDITIONS closures,
# which otherwise keep the weights alive) so each worker owns a clean GPU.
#
# NB: only the *parallel* branch is freed. On a single GPU we keep the model
# because run_conditions() runs in THIS process. The scalar neuron coords
# (L_star, top_overconf, …) survive either way — the run cell rebuilds the
# specs from them.
import gc, torch
_N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
if _N_GPUS > 1:
    # Both dicts hold factory closures over lm.layers → they pin the weights.
    for _cname in ('CONDITIONS', 'ALL_CONDITIONS'):
        try:
            del globals()[_cname]
        except KeyError:
            pass
    try:
        if 'lm' in dir() and lm is not None:
            for _attr in ('model', 'tokenizer', 'layers'):
                if hasattr(lm, _attr):
                    setattr(lm, _attr, None)
        lm = None
    except Exception as _e:
        print('main-model free skipped:', _e)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'Freed main model before parallel H6 ({_N_GPUS} GPUs → workers own them).')
else:
    print('Single GPU — keeping the main model for the sequential H6 path.')

In [ ]:
# === start a periodic mirror for the duration of the H6 run ================
# The benchmark writes per-worker shards to results/h6/_gpu{rank}/ and only
# the END-of-phase mirror would otherwise push them. A disconnect mid-run
# would then lose all partial shards → the next runtime restarts from zero.
# Mirroring every 3 min bounds the loss; restore is recursive so the partial
# _gpu* shards come back and each worker resumes positionally from its shard.
_mirror = None
if REMOTE:
    from src.persist import PeriodicMirror
    _mirror = PeriodicMirror(str(RESULTS), REMOTE, BACKEND, interval=180).start()
    print(f'Periodic mirror running every 180s → {REMOTE}')
else:
    print('local backend — no periodic mirror (results stay in this runtime).')

In [ ]:
# === H6 — run all conditions (resumable, multi-GPU if available) ===
import traceback
try:

    # Wall-time depends almost entirely on tokens generated per question. With the
    # early-stop criterion (halts just after "Answer: X") a question is a few
    # seconds, so:
    #   single 80GB GPU   1273×3  ≈ 1.5-3 h   (NF4/bf16 32B)
    #   4× A100-40 parallel 1273×6 ≈ 1.5-2 h  (each GPU ~318 items)
    # WITHOUT early-stop (the old default) every question ran to 512 tokens ≈ 25 s,
    # making this 20+ h — if you see that, your src/ is stale: re-run the env-check
    # cell (it git-resets to origin/main) to pull the fix.
    #
    # The probe cell above prints an honest estimate for YOUR config. If it says
    # >3 h, set `%env N_BENCH=300` and Run-all for a fast first pass (resumable).
    #
    # If multiple GPUs are visible we partition items round-robin and spawn
    # one worker process per GPU. Each worker loads its own model copy and
    # runs its slice through run_conditions. Main merges JSONLs.
    import time
    N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f'CUDA devices visible: {N_GPUS}')

    t0 = time.time()
    if N_GPUS > 1:
        # Multi-GPU data parallelism. Conditions are passed as JSON specs so each
        # worker can rebuild its own factories using its own lm.layers.
        from src.parallel import run_conditions_parallel
        condition_specs = {'baseline': None}
        condition_specs['h1_gate_anchor'] = dict(
            type='anchor', layer=L_star, neuron=N_star,
            m_star=float(m_star), d=float(anchor_d), k=1.0,
        )
        condition_specs['h5_ablate_overconf'] = dict(
            type='ablate', neurons=top_overconf,
        )
        # FAST or DEEP_FULL → run all six conditions in parallel.
        if H6_MODE in ('FAST', 'DEEP_FULL'):
            condition_specs['h3_zero_layer']      = dict(type='zero_mlp', layers=[int(critical)])
            condition_specs['h4_ablate_halluc']   = dict(type='ablate', neurons=top_halluc)
            condition_specs['h4_h5_combined']     = dict(type='ablate', neurons=combined)
        # Force 4-bit quant in workers regardless of USE_4BIT (which was set in
        # the env-check cell for *single*-GPU bf16). With N model copies in N
        # processes, 4-bit gives ~66 GB headroom per H100 vs 16 GB at bf16 —
        # OOM-proof on reasoning chains.
        run_conditions_parallel(
            model_name=MODEL_NAME, items=items, condition_specs=condition_specs,
            out_dir=H6_RESULTS, n_gpus=N_GPUS, token=os.environ.get('HF_TOKEN'),
            quantize_4bit=True,
        )
        # Re-build all_results from the merged jsonls so downstream cells work.
        from src.healthbench import BenchmarkRow
        all_results = {}
        for cond in condition_specs:
            rows = []
            path = H6_RESULTS / f'{cond}.jsonl'
            if path.exists():
                for line in path.read_text().splitlines():
                    if line.strip():
                        rows.append(BenchmarkRow.from_dict(json.loads(line)))
            all_results[cond] = rows
        CONDITIONS = condition_specs   # so downstream cells iterate the right keys
    else:
        all_results = run_conditions(
            lm, items, CONDITIONS, out_dir=H6_RESULTS, save_every=25,
        )
    print(f'\nDone in {(time.time() - t0)/60:.1f} min')

    import json
    summary = json.loads((H6_RESULTS / 'summary.json').read_text())
    print(f'\n{"condition":<22} {"acc":>6}  {"p_top1@ans":>10}  {"p_gold@ans":>10}  {"brier@ans":>10}  {"ans_found":>10}')
    print('-' * 80)
    for c, s in summary.items():
        print(f'{c:<22} {s["accuracy"]:>6.3f}  {s["mean_p_top1_at_answer"]:>10.4f}  '
              f'{s["mean_p_gold_at_answer"]:>10.4f}  {s["brier_at_answer"]:>10.4f}  '
              f'{s["answer_position_found_rate"]:>10.3f}')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === stop the periodic mirror (does one final sync) ========================
if _mirror is not None:
    _mirror.stop(final=True)
    print(f'Periodic mirror stopped after {_mirror.n_syncs} syncs (+ final).')

In [ ]:
# === H6 — sample reasoning per condition ===
import traceback
try:

    # Inspect how each intervention changes the reasoning chain on the same
    # question. Helpful when accuracy is similar but the answer's *justification*
    # shifts (e.g. anchor intervention preserves the letter but hedges more).
    import textwrap
    EX = 3
    for cond, rows in all_results.items():
        print(f'\n========== {cond} (sample of {EX}) ==========')
        for r in rows[:EX]:
            verdict = 'OK ' if r.correct else 'ERR'
            print(f'\n[{verdict}] gold={r.gold}  pred={r.predicted}  '
                  f'p@ans={r.p_top1_at_answer:.3f}  p_gold@ans={r.p_gold_at_answer:.3f}  '
                  f'ans_found={r.answer_pos_found}')
            print('Q :', textwrap.shorten(r.question, 200))
            rsn = r.reasoning or '(no parsed reasoning — raw_output:)'
            print('R :', textwrap.shorten(rsn or r.raw_output, 300))
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === H6 — comparison table + delta plot ===
import traceback
try:

    import csv, matplotlib.pyplot as plt, numpy as np
    rows = []
    with open(H6_RESULTS / 'comparison.csv') as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
    print(f'Per-question comparison: {len(rows)} rows in {H6_RESULTS / "comparison.csv"}')

    # Delta accuracy vs baseline.
    conds = [c for c in CONDITIONS.keys() if c != 'baseline']
    base_acc = sum(int(r['baseline_correct'] or 0) for r in rows) / max(1, len(rows))
    print(f'\nBaseline accuracy: {base_acc:.3f}')
    for c in conds:
        acc = sum(int(r[f'{c}_correct'] or 0) for r in rows) / max(1, len(rows))
        print(f'  {c:<20}: {acc:.3f}  ({acc - base_acc:+.3f})')

    # Calibration at the ANSWER token (not the first generated token).
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for c in CONDITIONS:
        p = np.array([float(r[f'{c}_p_top1_answer']) for r in rows if r[f'{c}_p_top1_answer']])
        correct = np.array([int(r[f'{c}_correct'] or 0) for r in rows if r[f'{c}_p_top1_answer']])
        if len(p) == 0:
            continue
        axes[0].hist(p, bins=20, histtype='step', label=c, alpha=0.8, linewidth=1.5)
        bins = np.linspace(0, 1, 11)
        bin_idx = np.digitize(p, bins) - 1
        means = [correct[bin_idx == b].mean() if (bin_idx == b).any() else np.nan for b in range(10)]
        axes[1].plot((bins[:-1] + bins[1:]) / 2, means, marker='o', label=c, alpha=0.8)
    axes[0].set_xlabel('p_top1 at the answer-letter position'); axes[0].set_ylabel('# questions')
    axes[0].set_title('Confidence at Answer:'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[1].plot([0, 1], [0, 1], 'k--', lw=0.5, label='perfect calibration')
    axes[1].set_xlabel('predicted p_top1 @ answer'); axes[1].set_ylabel('empirical accuracy')
    axes[1].set_title('Reliability diagram (answer-position)'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(H6_RESULTS / 'reliability.png', dpi=140); plt.show()
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === consensus-flip analyzer ===
import traceback
try:

    from src.consensus import analyze, summarize

    conds = list(CONDITIONS.keys())
    rows = analyze(H6_RESULTS / 'comparison.csv', conds)
    report = summarize(rows, conds)

    print(f"Total questions          : {report['n_total']}")
    print(f"Baseline wrong           : {report['n_baseline_wrong']}")
    print(f"Consensus-flip cases     : {report['n_consensus_flips']}")
    print()
    print(f'{"condition":<22}  flips fixed  on-flips %   any baseline-wrong fixed  any-rate %')
    print('-' * 92)
    for c, info in report['fix_rates'].items():
        print(f'{c:<22}  {info["on_flips"]:>11}  {100*info["on_flips_rate"]:>9.1f}%  '
              f'{info["on_any_baseline_wrong"]:>23}  {100*info["on_any_rate"]:>8.1f}%')

    # Per-row dump for offline inspection.
    import json as _json
    (H6_RESULTS / 'consensus_flip.json').write_text(
        _json.dumps({'report': report,
                     'rows': [r.__dict__ for r in rows]}, indent=2))
    print('\nWrote', H6_RESULTS / 'consensus_flip.json')
except Exception:
    print('!!!!!!!!!! CELL FAILED — full traceback below !!!!!!!!!!')
    traceback.print_exc()
    raise

In [ ]:
# === persist: mirror results/ back to the shared backend ===================
# Run this LAST so the next phase's runtime can restore what this phase made.
# (`_sync` was defined in the restore cell — same guards apply.)
if REMOTE:
    if BACKEND == 'drive':
        Path(REMOTE).mkdir(parents=True, exist_ok=True)
    _sync(str(RESULTS), REMOTE, BACKEND, 'mirror')       # local -> remote
    print('Mirrored results/ →', REMOTE)
else:
    print('local backend — results stay in /content/results only this session.')